In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path
import sqlalchemy as sa

In [2]:
# Evita notación científica al mostrar DataFrames
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [3]:
#Creamos la ruta a la bbdd
ruta_db = Path('..') / 'datos'/ 'brutos' / 'ecommerce.db'

#Creamos la conexión a la bbdd
engine = sa.create_engine(f'sqlite:///{ruta_db.resolve()}')

#Probamos la conexión
with engine.connect() as conn:
    print('Conexión exitosa.', conn)

    

Conexión exitosa. <sqlalchemy.engine.base.Connection object at 0x000001545A34BE60>


In [4]:
# Inspeccionamos la bbdd y extraemos todas las tablas.
from sqlalchemy import inspect
inspector = inspect(engine)
tablas = inspector.get_table_names()
print("Tablas en la base de datos: ", tablas)

Tablas en la base de datos:  ['2019-Dec', '2019-Nov', '2019-Oct', '2020-Feb', '2020-Jan']


In [6]:
oct = pd.read_sql('2019-Oct', engine)
nov = pd.read_sql('2019-Nov', engine)
dic = pd.read_sql('2019-Dec', engine)
ene = pd.read_sql('2020-Jan', engine)
feb = pd.read_sql('2020-Feb', engine)

In [7]:
# Comprobamos que todas las tablas tienen la misma estructura y mismo número de columnas.
print(oct.shape, nov.shape, dic.shape, ene.shape, feb.shape)

(407925, 10) (462833, 10) (351304, 10) (443224, 10) (429790, 10)


In [8]:
#Comprobamos que todas las tablas tienen la misma estructura y mismo número de columnas.

print(oct.columns)
print(nov.columns)
print(dic.columns)
print(ene.columns)
print(feb.columns)

Index(['index', 'event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='str')
Index(['index', 'event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='str')
Index(['index', 'event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='str')
Index(['index', 'event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='str')
Index(['index', 'event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='str')


In [9]:
df = pd.concat([oct, nov, dic, ene, feb], axis=0)
df.info()

<class 'pandas.DataFrame'>
Index: 2095076 entries, 0 to 429789
Data columns (total 10 columns):
 #   Column         Dtype  
---  ------         -----  
 0   index          int64  
 1   event_time     str    
 2   event_type     str    
 3   product_id     int64  
 4   category_id    int64  
 5   category_code  str    
 6   brand          str    
 7   price          float64
 8   user_id        int64  
 9   user_session   str    
dtypes: float64(1), int64(4), str(5)
memory usage: 175.8 MB


In [10]:
df.shape

(2095076, 10)

CALIDAD DE DATOS

In [11]:
df.head()

,index,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,68,2019-10-01 00:01:46 UTC,view,5843665,1487580005092295511,NaN,f.o.x,9.44,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361
1,72,2019-10-01 00:01:55 UTC,cart,5868461,1487580013069861041,NaN,italwax,3.57,514753614,e2fecb2d-22d0-df2c-c661-15da44b3ccf1
2,95,2019-10-01 00:02:50 UTC,view,5877456,1487580006300255120,NaN,jessnail,122.22,527418424,86e77869-afbc-4dff-9aa2-6b7dd8c90770
3,122,2019-10-01 00:03:41 UTC,view,5649270,1487580013749338323,NaN,concept,6.19,555448072,b5f72ceb-0730-44de-a932-d16db62390df
4,124,2019-10-01 00:03:44 UTC,view,18082,1487580005411062629,NaN,cnd,16.03,552006247,2d8f304b-de45-4e59-8f40-50c603843fe5


In [12]:
# Como event_time no la ha cogido como datetime, lo convertimos.
df['event_time'] = pd.to_datetime(df['event_time'])
df.head()

,index,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,68,2019-10-01 00:01:46+00:00,view,5843665,1487580005092295511,NaN,f.o.x,9.44,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361
1,72,2019-10-01 00:01:55+00:00,cart,5868461,1487580013069861041,NaN,italwax,3.57,514753614,e2fecb2d-22d0-df2c-c661-15da44b3ccf1
2,95,2019-10-01 00:02:50+00:00,view,5877456,1487580006300255120,NaN,jessnail,122.22,527418424,86e77869-afbc-4dff-9aa2-6b7dd8c90770
3,122,2019-10-01 00:03:41+00:00,view,5649270,1487580013749338323,NaN,concept,6.19,555448072,b5f72ceb-0730-44de-a932-d16db62390df
4,124,2019-10-01 00:03:44+00:00,view,18082,1487580005411062629,NaN,cnd,16.03,552006247,2d8f304b-de45-4e59-8f40-50c603843fe5


In [13]:
df.info()

<class 'pandas.DataFrame'>
Index: 2095076 entries, 0 to 429789
Data columns (total 10 columns):
 #   Column         Dtype              
---  ------         -----              
 0   index          int64              
 1   event_time     datetime64[us, UTC]
 2   event_type     str                
 3   product_id     int64              
 4   category_id    int64              
 5   category_code  str                
 6   brand          str                
 7   price          float64            
 8   user_id        int64              
 9   user_session   str                
dtypes: datetime64[us, UTC](1), float64(1), int64(4), str(4)
memory usage: 175.8 MB


In [14]:
# La variable index no es necesaria. La eliminamos.
df = df.drop(columns=['index'])

In [15]:
# Traducimos los nombres de las columnas a español.
df.columns = ['fecha_hora', 'evento', 'producto', 'categoria', 'categoria_cod',
 'marca', 'precio', 'usuario', 'sesion']
df.head()



,fecha_hora,evento,producto,categoria,categoria_cod,marca,precio,usuario,sesion
0,2019-10-01 00:01:46+00:00,view,5843665,1487580005092295511,NaN,f.o.x,9.44,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361
1,2019-10-01 00:01:55+00:00,cart,5868461,1487580013069861041,NaN,italwax,3.57,514753614,e2fecb2d-22d0-df2c-c661-15da44b3ccf1
2,2019-10-01 00:02:50+00:00,view,5877456,1487580006300255120,NaN,jessnail,122.22,527418424,86e77869-afbc-4dff-9aa2-6b7dd8c90770
3,2019-10-01 00:03:41+00:00,view,5649270,1487580013749338323,NaN,concept,6.19,555448072,b5f72ceb-0730-44de-a932-d16db62390df
4,2019-10-01 00:03:44+00:00,view,18082,1487580005411062629,NaN,cnd,16.03,552006247,2d8f304b-de45-4e59-8f40-50c603843fe5


In [16]:
# ANÁLISIS DE NULOS
# Al hacer info no se veían los nulos. Los saco a parte.
df.isnull().sum()

fecha_hora             0
evento                 0
producto               0
categoria              0
categoria_cod    2060411
marca             891646
precio                 0
usuario                0
sesion               506
dtype: int64

In [17]:
df.categoria_cod.value_counts()

categoria_cod
appliances.environment.vacuum             14066
stationery.cartrige                        6037
apparel.glove                              5671
furniture.living_room.cabinet              2900
accessories.bag                            2419
furniture.bathroom.bath                    2245
appliances.personal.hair_cutter             460
accessories.cosmetic_bag                    421
appliances.personal.massager                359
appliances.environment.air_conditioner       62
furniture.living_room.chair                  25
Name: count, dtype: int64

Dado que tenemos la categoria_cod tiene casi el 100% de nulos, y además tenemos categoria sin nulos, eliminamos categori_cod, pues sería redundante.
También eliminamos marca porque el 50% son nulos.

In [18]:
df = df.drop(columns=['categoria_cod', 'marca'])

En cuanto a sesión, es una variable muy relevante y no podemos imputarla. Eliminaremos esos 506 registros.

In [19]:
df.dropna(subset = ['sesion'], inplace=True)

In [20]:
# Comprobamos
df.isnull().sum()

fecha_hora    0
evento        0
producto      0
categoria     0
precio        0
usuario       0
sesion        0
dtype: int64

In [21]:
df.info(show_counts=True)

<class 'pandas.DataFrame'>
Index: 2094570 entries, 0 to 429789
Data columns (total 7 columns):
 #   Column      Non-Null Count    Dtype              
---  ------      --------------    -----              
 0   fecha_hora  2094570 non-null  datetime64[us, UTC]
 1   evento      2094570 non-null  str                
 2   producto    2094570 non-null  int64              
 3   categoria   2094570 non-null  int64              
 4   precio      2094570 non-null  float64            
 5   usuario     2094570 non-null  int64              
 6   sesion      2094570 non-null  str                
dtypes: datetime64[us, UTC](1), float64(1), int64(3), str(2)
memory usage: 127.8 MB


ANÁLISIS VARIABLES NUMÉRICAS

In [22]:
# Realmente la única variable numérica es el precio. El resto son categóricas.
# Apreciamos que la mediana del precio es 4. La media es de 8.4
# Ojo que hay productos con preio negativo. Probablemente son devoluciones.

df.describe()

,producto,categoria,precio,usuario
count,"2,094,570.00","2,094,570.00","2,094,570.00","2,094,570.00"
mean,"5,487,103.56","1,553,112,489,392,098,048.00",8.42,"521,077,545.56"
std,"1,300,923.90","167,907,497,920,480,608.00",19.14,"87,553,855.76"
min,"3,752.00","1,487,580,004,807,082,752.00",-47.62,"4,661,182.00"
25%,"5,724,652.00","1,487,580,005,754,995,456.00",2.05,"480,613,387.00"
50%,"5,811,665.00","1,487,580,008,246,412,288.00",4.00,"553,341,613.00"
75%,"5,858,353.00","1,487,580,013,489,291,520.00",6.86,"578,406,571.00"
max,"5,932,595.00","2,242,903,426,784,559,104.00",327.78,"622,087,993.00"


In [23]:
print("Precios 0; ", df[df.precio == 0].shape[0])
print("Precios negativos: ", df[df.precio < 0].shape[0])

# Vemos que hay 20533 productos con precio 0.
# Eliminamos esos 20533 productos.

# Vemos que hay 11 productos con precio negativo.
# Probablemente sean devoluciones.
# Eliminamos esos 11 productos.

df = df[df.precio > 0]
df.describe()


Precios 0;  20533
Precios negativos:  11


,producto,categoria,precio,usuario
count,"2,074,026.00","2,074,026.00","2,074,026.00","2,074,026.00"
mean,"5,485,203.09","1,553,192,964,541,272,064.00",8.50,"521,758,909.23"
std,"1,304,219.20","168,128,659,843,838,048.00",19.21,"87,354,480.68"
min,"3,752.00","1,487,580,004,807,082,752.00",0.05,"4,661,182.00"
25%,"5,724,633.00","1,487,580,005,754,995,456.00",2.06,"482,863,572.00"
50%,"5,811,652.00","1,487,580,008,246,412,288.00",4.06,"553,690,395.00"
75%,"5,858,221.00","1,487,580,013,489,291,520.00",6.98,"578,763,499.00"
max,"5,932,585.00","2,242,903,426,784,559,104.00",327.78,"622,087,993.00"


In [24]:
# ANALISIS VARIABLES CATEGÓRICAS
# Tiene sentido analizar evento, producto y categoria.
# El resto son variables de tipo identificadoras.

# Evento parece correcto.
df.evento.value_counts()



evento
view                961558
cart                574547
remove_from_cart    410357
purchase            127564
Name: count, dtype: int64

In [25]:
# Hay muchísimis productos. No vale la pena hacer un value_counts.
df.producto.nunique()

45327

In [26]:
# Hay muchísimas categorías. No vale la pena hacer un value_counts.
df.categoria.nunique()

508

In [27]:
df.categoria.value_counts(normalize=True)

categoria
1487580007675986893   0.05
1487580005595612013   0.04
1487580005092295511   0.04
1487580005671109489   0.03
1602943681873052386   0.03
                      ... 
1487580013363462335   0.00
1487580008204469224   0.00
1487580011559911545   0.00
1487580009857024046   0.00
2053031020655018687   0.00
Name: proportion, Length: 508, dtype: float64

CREACIÓN VARIABLES

In [28]:
# Variable fecha como índice
df = df.set_index('fecha_hora')
# Extraemos el año, mes, día, hora y minuto.
df['año'] = df.index.year
df['mes'] = df.index.month
df['dia'] = df.index.day
df['hora'] = df.index.hour
df['minuto'] = df.index.minute

# Creamos la variable dia_semana, nombre_mes, nombre_dia, trimestre
df['dia_semana'] = df.index.dayofweek +1  # 1 = lunes, 7 = domingo
df['nombre_mes'] = df.index.strftime('%B')
df['nombre_dia'] = df.index.strftime('%A')
df['trimestre'] = df.index.quarter

# Creamos la variable fecha sin la hora
df['fecha'] = df.index.date




In [29]:
df.head()

,evento,producto,categoria,precio,usuario,sesion,año,mes,dia,hora,minuto,dia_semana,nombre_mes,nombre_dia,trimestre,fecha
fecha_hora,,,,,,,,,,,,,,,,
2019-10-01 00:01:46+00:00,view,5843665,1487580005092295511,9.44,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361,2019,10,1,0,1,2,October,Tuesday,4,2019-10-01
2019-10-01 00:01:55+00:00,cart,5868461,1487580013069861041,3.57,514753614,e2fecb2d-22d0-df2c-c661-15da44b3ccf1,2019,10,1,0,1,2,October,Tuesday,4,2019-10-01
2019-10-01 00:02:50+00:00,view,5877456,1487580006300255120,122.22,527418424,86e77869-afbc-4dff-9aa2-6b7dd8c90770,2019,10,1,0,2,2,October,Tuesday,4,2019-10-01
2019-10-01 00:03:41+00:00,view,5649270,1487580013749338323,6.19,555448072,b5f72ceb-0730-44de-a932-d16db62390df,2019,10,1,0,3,2,October,Tuesday,4,2019-10-01
2019-10-01 00:03:44+00:00,view,18082,1487580005411062629,16.03,552006247,2d8f304b-de45-4e59-8f40-50c603843fe5,2019,10,1,0,3,2,October,Tuesday,4,2019-10-01


In [30]:
# AÑADIMOS LOS FESTIVOS 
# para ello instalamos la librería holidays
# !pip install holidays
import holidays

# Hacemos una prueba con el calendario de España y creamos el diccionario f (festivos)
f = holidays.ES(years=2025)
print(f)
for fecha, fiesta in f.items():
    print(fecha, fiesta)




{datetime.date(2025, 1, 1): 'Año Nuevo', datetime.date(2025, 1, 6): 'Epifanía del Señor', datetime.date(2025, 4, 18): 'Viernes Santo', datetime.date(2025, 5, 1): 'Fiesta del Trabajo', datetime.date(2025, 8, 15): 'Asunción de la Virgen', datetime.date(2025, 11, 1): 'Todos los Santos', datetime.date(2025, 12, 6): 'Día de la Constitución Española', datetime.date(2025, 12, 8): 'Inmaculada Concepción', datetime.date(2025, 12, 25): 'Natividad del Señor'}
2025-01-01 Año Nuevo
2025-01-06 Epifanía del Señor
2025-04-18 Viernes Santo
2025-05-01 Fiesta del Trabajo
2025-08-15 Asunción de la Virgen
2025-11-01 Todos los Santos
2025-12-06 Día de la Constitución Española
2025-12-08 Inmaculada Concepción
2025-12-25 Natividad del Señor


In [31]:
# como funciona bien, lo aplicamos a Rusia para los años 2019 y 2020
f_ru = holidays.RU(years=[2019, 2020])
for fecha, fiesta in f_ru.items():
    print(fecha, fiesta)


2019-01-01 Новогодние каникулы
2019-01-02 Новогодние каникулы
2019-01-03 Новогодние каникулы
2019-01-04 Новогодние каникулы
2019-01-05 Новогодние каникулы
2019-01-06 Новогодние каникулы
2019-01-08 Новогодние каникулы
2019-01-07 Рождество Христово
2019-02-23 День защитника Отечества
2019-03-08 Международный женский день
2019-05-01 Праздник Весны и Труда
2019-05-09 День Победы
2019-06-12 День России
2019-11-04 День народного единства
2019-05-02 Выходной (перенесено с 05.01.2019)
2019-05-03 Выходной (перенесено с 06.01.2019)
2019-05-10 Выходной (перенесено с 23.02.2019)
2020-01-01 Новогодние каникулы
2020-01-02 Новогодние каникулы
2020-01-03 Новогодние каникулы
2020-01-04 Новогодние каникулы
2020-01-05 Новогодние каникулы
2020-01-06 Новогодние каникулы
2020-01-08 Новогодние каникулы
2020-01-07 Рождество Христово
2020-02-23 День защитника Отечества
2020-03-08 Международный женский день
2020-05-01 Праздник Весны и Труда
2020-05-09 День Победы
2020-06-12 День России
2020-11-04 День народного

In [32]:
# creamos una variable festivo en df que toma valor 1 si la fecha es festiva y 0 en caso contrario
df['festivo'] = df.fecha.apply(lambda x: 1 if x in f_ru else 0)
df['festivo'].value_counts()

festivo
0    1940833
1     133193
Name: count, dtype: int64

In [33]:
df.head()

,evento,producto,categoria,precio,usuario,sesion,año,mes,dia,hora,minuto,dia_semana,nombre_mes,nombre_dia,trimestre,fecha,festivo
fecha_hora,,,,,,,,,,,,,,,,,
2019-10-01 00:01:46+00:00,view,5843665,1487580005092295511,9.44,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361,2019,10,1,0,1,2,October,Tuesday,4,2019-10-01,0
2019-10-01 00:01:55+00:00,cart,5868461,1487580013069861041,3.57,514753614,e2fecb2d-22d0-df2c-c661-15da44b3ccf1,2019,10,1,0,1,2,October,Tuesday,4,2019-10-01,0
2019-10-01 00:02:50+00:00,view,5877456,1487580006300255120,122.22,527418424,86e77869-afbc-4dff-9aa2-6b7dd8c90770,2019,10,1,0,2,2,October,Tuesday,4,2019-10-01,0
2019-10-01 00:03:41+00:00,view,5649270,1487580013749338323,6.19,555448072,b5f72ceb-0730-44de-a932-d16db62390df,2019,10,1,0,3,2,October,Tuesday,4,2019-10-01,0
2019-10-01 00:03:44+00:00,view,18082,1487580005411062629,16.03,552006247,2d8f304b-de45-4e59-8f40-50c603843fe5,2019,10,1,0,3,2,October,Tuesday,4,2019-10-01,0


In [34]:
special_dates = {
    'dia_singles_11nov': [pd.Timestamp('2019-11-11')],           # Singles' Day
    'dia_unidad_nacional_4nov': [pd.Timestamp('2019-11-04')],   # Día de la Unidad Nacional
    'black_friday_2019': [pd.Timestamp('2019-11-29')],          # Black Friday 2019
    'cyber_monday_2019': [pd.Timestamp('2019-12-02')],          # Cyber Monday 2019
    'periodo_fin_ano_nuevo_ano': pd.date_range('2019-12-31','2020-01-08'),  # Vacaciones de Año Nuevo en Rusia
    'navidad_ortodoxa_7ene': [pd.Timestamp('2020-01-07')],      # Navidad ortodoxa
    'defensor_patria_23feb': [pd.Timestamp('2020-02-23')]       # Día del Defensor de la Patria
}

In [35]:
# Crear indicadores 0/1 para cada fecha especial en special_dates

for nombre, dias in special_dates.items():
    # Normalizar la lista de días a objetos date
    dias_list = list(dias)
    dias_ts = pd.to_datetime(dias_list)
    dias_dates = set(d.date() for d in dias_ts)

    # Crear la columna indicadora (1 si la fecha del registro está en dias_dates)
    df[nombre] = df['fecha'].apply(lambda x: 1 if x in dias_dates else 0)

In [36]:
df.head()

,evento,producto,categoria,precio,usuario,sesion,año,mes,dia,hora,...,trimestre,fecha,festivo,dia_singles_11nov,dia_unidad_nacional_4nov,black_friday_2019,cyber_monday_2019,periodo_fin_ano_nuevo_ano,navidad_ortodoxa_7ene,defensor_patria_23feb
fecha_hora,,,,,,,,,,,,,,,,,,,,,
2019-10-01 00:01:46+00:00,view,5843665,1487580005092295511,9.44,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361,2019,10,1,0,...,4,2019-10-01,0,0,0,0,0,0,0,0
2019-10-01 00:01:55+00:00,cart,5868461,1487580013069861041,3.57,514753614,e2fecb2d-22d0-df2c-c661-15da44b3ccf1,2019,10,1,0,...,4,2019-10-01,0,0,0,0,0,0,0,0
2019-10-01 00:02:50+00:00,view,5877456,1487580006300255120,122.22,527418424,86e77869-afbc-4dff-9aa2-6b7dd8c90770,2019,10,1,0,...,4,2019-10-01,0,0,0,0,0,0,0,0
2019-10-01 00:03:41+00:00,view,5649270,1487580013749338323,6.19,555448072,b5f72ceb-0730-44de-a932-d16db62390df,2019,10,1,0,...,4,2019-10-01,0,0,0,0,0,0,0,0
2019-10-01 00:03:44+00:00,view,18082,1487580005411062629,16.03,552006247,2d8f304b-de45-4e59-8f40-50c603843fe5,2019,10,1,0,...,4,2019-10-01,0,0,0,0,0,0,0,0


In [39]:
# Ordenamos las variables según el orden lógico del proceso de compra
variables = df.columns.to_list()
variables

['evento',
 'producto',
 'categoria',
 'precio',
 'usuario',
 'sesion',
 'año',
 'mes',
 'dia',
 'hora',
 'minuto',
 'dia_semana',
 'nombre_mes',
 'nombre_dia',
 'trimestre',
 'fecha',
 'festivo',
 'dia_singles_11nov',
 'dia_unidad_nacional_4nov',
 'black_friday_2019',
 'cyber_monday_2019',
 'periodo_fin_ano_nuevo_ano',
 'navidad_ortodoxa_7ene',
 'defensor_patria_23feb']

In [44]:
primera_parte = ['usuario', 'sesion','evento','categoria','producto','precio']
resto = variables[6:]
orden = primera_parte+resto
orden

['usuario',
 'sesion',
 'evento',
 'categoria',
 'producto',
 'precio',
 'año',
 'mes',
 'dia',
 'hora',
 'minuto',
 'dia_semana',
 'nombre_mes',
 'nombre_dia',
 'trimestre',
 'fecha',
 'festivo',
 'dia_singles_11nov',
 'dia_unidad_nacional_4nov',
 'black_friday_2019',
 'cyber_monday_2019',
 'periodo_fin_ano_nuevo_ano',
 'navidad_ortodoxa_7ene',
 'defensor_patria_23feb']

In [46]:
df = df[orden]
df

,usuario,sesion,evento,categoria,producto,precio,año,mes,dia,hora,...,trimestre,fecha,festivo,dia_singles_11nov,dia_unidad_nacional_4nov,black_friday_2019,cyber_monday_2019,periodo_fin_ano_nuevo_ano,navidad_ortodoxa_7ene,defensor_patria_23feb
fecha_hora,,,,,,,,,,,,,,,,,,,,,
2019-10-01 00:01:46+00:00,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361,view,1487580005092295511,5843665,9.44,2019,10,1,0,...,4,2019-10-01,0,0,0,0,0,0,0,0
2019-10-01 00:01:55+00:00,514753614,e2fecb2d-22d0-df2c-c661-15da44b3ccf1,cart,1487580013069861041,5868461,3.57,2019,10,1,0,...,4,2019-10-01,0,0,0,0,0,0,0,0
2019-10-01 00:02:50+00:00,527418424,86e77869-afbc-4dff-9aa2-6b7dd8c90770,view,1487580006300255120,5877456,122.22,2019,10,1,0,...,4,2019-10-01,0,0,0,0,0,0,0,0
2019-10-01 00:03:41+00:00,555448072,b5f72ceb-0730-44de-a932-d16db62390df,view,1487580013749338323,5649270,6.19,2019,10,1,0,...,4,2019-10-01,0,0,0,0,0,0,0,0
2019-10-01 00:03:44+00:00,552006247,2d8f304b-de45-4e59-8f40-50c603843fe5,view,1487580005411062629,18082,16.03,2019,10,1,0,...,4,2019-10-01,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-02-29 23:58:49+00:00,147995998,5ff96629-3627-493e-a25b-5a871ec78c90,cart,1487580006317032337,5815662,0.92,2020,2,29,23,...,1,2020-02-29,0,0,0,0,0,0,0,0
2020-02-29 23:58:57+00:00,147995998,5ff96629-3627-493e-a25b-5a871ec78c90,view,1487580006317032337,5815665,0.59,2020,2,29,23,...,1,2020-02-29,0,0,0,0,0,0,0,0
2020-02-29 23:59:05+00:00,147995998,5ff96629-3627-493e-a25b-5a871ec78c90,cart,1487580006317032337,5815665,0.59,2020,2,29,23,...,1,2020-02-29,0,0,0,0,0,0,0,0


In [47]:
df.to_pickle(Path('..') / 'datos' /'intermedios'/'tablon_analitico.pickle')